# 9장 - 텍스트+화자 매칭

원본 파일: `chap09/merge_speaker_text.py`

In [1]:
import os
import pandas as pd
from whisper_local import whisper_stt
from speaker_diarization import diarize_to_rttm, rttm_to_grouped_csv

BASE_DIR = (os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd())

/Users/seminy/Desktop/Main/Git/AI_Bootcamp/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/seminy/Desktop/Main/Git/AI_Bootcamp/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def match_speaker(text_start, text_end, df_speakers):
    """발화 텍스트 구간과 가장 많이 겹치는 화자를 찾아서 반환"""
    best_speaker = None
    best_overlap = 0

    for _, row in df_speakers.iterrows():
        overlap = min(text_end, row['end']) - max(text_start, row['start'])
        if overlap > best_overlap:
            best_overlap = overlap
            best_speaker = row['speaker_id']

    return best_speaker

In [3]:
def transcribe_with_speakers(audio_file_path: str, output_csv_path: str):
    base_name = os.path.splitext(os.path.basename(audio_file_path))[0]
    rttm_path = os.path.join(BASE_DIR, 'data', f'{base_name}.rttm')
    speaker_csv_path = os.path.join(BASE_DIR, 'data', f'{base_name}_rttm.csv')
    text_csv_path = os.path.join(BASE_DIR, 'data', f'{base_name}_text.csv')

    # 1) 화자 분리 (이미 결과가 있으면 재사용)
    if not os.path.exists(speaker_csv_path):
        diarize_to_rttm(audio_file_path, rttm_path)
        df_speakers = rttm_to_grouped_csv(rttm_path, speaker_csv_path)
    else:
        df_speakers = pd.read_csv(speaker_csv_path)

    # 2) 음성을 텍스트로 변환
    _, df_text = whisper_stt(audio_file_path, text_csv_path)

    # 3) 각 발화 텍스트 구간에 가장 많이 겹치는 화자를 매칭
    df_text['speaker_id'] = df_text.apply(
        lambda row: match_speaker(row['start'], row['end'], df_speakers), axis=1
    )

    df_text.to_csv(output_csv_path, index=False, sep='|')
    return df_text

## 실행 / 테스트

In [4]:
audio_path = os.path.join(BASE_DIR, 'data', '싼기타_비싼기타.mp3')
output_path = os.path.join(BASE_DIR, 'data', '싼기타_비싼기타_merged.csv')

In [5]:
df = transcribe_with_speakers(audio_path, output_path)
print(df)

`torch_dtype` is deprecated! Use `dtype` instead!
`torch_dtype` is deprecated! Use `dtype` instead!
Device set to use cpu
Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).
Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to E

     start     end                                               text  \
0     0.00    3.84                                 지금부터 저랑 역하의급을 합시다.   
1     3.84    6.96                          역하의급을 스탠드인 커뮤니티 스타일로 할 건데   
2     6.96   10.32                         토론을 하면서 자연스럽게 대화하는 용식을 하면서   
3    10.32   11.92                                      코매드를 진행해 봅시다.   
4    11.92   17.26       그래서 좀 재미있고 자연스럽고 유머러스하게 저랑 대화를 하시면 내가 잘 안스럽게   
..     ...     ...                                                ...   
76  407.00  409.00                                        뭐 하는 것 같은데?   
77  423.60  424.72  네, 정말 괜찮습니다. 즐겁게 대환하는 거 있었어요. 계속해서 이해하고 나누고 싶으...   
78  424.72  427.04                               언제든 다시 이야기 나누고 싶으실 때   
79  427.04  428.00                        편하게 말씀해주세요. 감사합니다. 수고하셨습니다.   
80  428.00  430.00                                             감사합니다.   

    speaker_id  
0   SPEAKER_01  
1   SPEAKER_01  
2   SPEAKER_01  
3   SPEAKER_01  
4   SPEAKER_01  
..         ...  
76  